In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Stage 2 Fairness

This notebook is reserved for fairness diagnostics on Stage 2 dropout-point prediction. The main question is whether the sequence model behaves differently across demographic groups or school-related categories after accounting for the behavioral trajectory setup.

Useful diagnostics include group support, dropout-point prevalence, thresholded error rates, calibration-style comparisons, and sensitivity to seasonal windows such as summer. Small groups and `Unknown` categories should be reported carefully rather than over-interpreted.


## Setup


In [3]:
import os
from pathlib import Path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

import torch

try:
    import lightgbm as lgb
except ModuleNotFoundError:
    lgb = None

PROJECT_ROOT = Path.cwd()

from src.preprocess import build_stage2_dataset
from src import model_utils as lstm
from src import sequence_utils as seq
from src.metrics_utils import validation_tuned_binary_metrics

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
SEQUENCE_LENGTH = 12
BATCH_SIZE = 64
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [5]:
stage2 = build_stage2_dataset(DATA_DIR)
df_full = stage2.df_full
feature_cols = stage2.feature_cols
users_clean = stage2.users
user_features = stage2.user_features

print(f"timeline rows: {len(df_full):,}")
print(f"users: {df_full['user_id'].nunique():,}")
print(f"features: {len(feature_cols)}")
print(f"labeled rows: {df_full['is_dropout_point'].notna().sum():,}")
print(feature_cols)

df_full.head()

timeline rows: 408,377
users: 22,470
features: 17
labeled rows: 342,844
['n_events', 'n_active_days', 'mean_hour', 'n_click_events', 'n_view_events', 'n_sessions', 'n_topics_event', 'n_transactions', 'correct_rate', 'partial_rate', 'mean_evaluation_score', 'avg_response_time', 'n_documents', 'n_topics_transaction', 'activity_score', 'year', 'day']


,user_id,relative_week,n_events,n_active_days,mean_hour,n_click_events,n_view_events,n_sessions,n_topics_event,n_transactions,...,avg_response_time,n_documents,n_topics_transaction,activity_score,week_start,year,day,is_active,is_dropout_point,is_summer
0,387604,0,2.0,2.0,9.0,0.0,2.0,0.0,0.0,2.0,...,0.0,1.0,0.0,6.0,2021-05-22 05:12:11.249,2021,142,1,0.0,False
1,387604,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-05-29 05:12:11.249,2021,149,0,0.0,False
2,387604,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-06-05 05:12:11.249,2021,156,0,0.0,False
3,387604,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-06-12 05:12:11.249,2021,163,0,0.0,False
4,387604,4,7.0,1.0,8.0,4.0,3.0,1.0,1.0,0.0,...,0.0,0.0,0.0,14.0,2021-06-19 05:12:11.249,2021,170,1,0.0,False


In [6]:
train_df, val_df, test_df, scaler = seq.split_and_scale_by_users(
    df_full,
    feature_cols,
    test_size=0.15,
    val_size=0.20,
    random_state=RANDOM_STATE,
)
print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

train: (275745, 23)
val:   (70920, 23)
test:  (61712, 23)


In [7]:
loaders, _ = lstm.make_sequence_loaders(
    train_df,
    val_df,
    test_df,
    feature_cols,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
)
train_loader, val_loader, test_loader = loaders
print("sequence batches:", len(train_loader), len(val_loader), len(test_loader))

sequence batches: 2250 591 506


In [8]:
def run_experiment_for_fairness(name, cols, demographics=False, drop_summer_test=False):
    local_train_df, local_val_df, local_test_df, _ = seq.split_and_scale_by_users(
        df_full,
        cols,
        test_size=0.15,
        val_size=0.20,
        random_state=RANDOM_STATE,
    )
    loaders, demo_sizes = lstm.make_sequence_loaders(
        local_train_df,
        local_val_df,
        local_test_df,
        cols,
        sequence_length=SEQUENCE_LENGTH,
        batch_size=BATCH_SIZE,
        users=users_clean,
        demographics=demographics,
        drop_summer_test=drop_summer_test,
    )
    train_loader, val_loader, test_loader = loaders

    if demographics:
        model = lstm.LSTMWithDemographics(
            input_size=len(cols),
            n_gender=demo_sizes["n_gender"],
            n_canton=demo_sizes["n_canton"],
            n_class=demo_sizes["n_class"],
        ).to(device)
    else:
        model = lstm.LSTMModel(input_size=len(cols)).to(device)

    model, history = lstm.train_lstm_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=EPOCHS,
        patience=PATIENCE,
        learning_rate=LEARNING_RATE,
        demographics=demographics,
    )
    metrics, test_prob, test_y = lstm.evaluate_lstm_model(
        model,
        val_loader,
        test_loader,
        device,
        demographics=demographics,
    )

    result = {
        "experiment": name,
        "n_features": len(cols),
        "demographics": demographics,
        "drop_summer_test": drop_summer_test,
        **metrics,
    }
    return model, test_loader

## Fairness tests

In [10]:
from src.fairness_utils_stage2 import *

### Model without demographic features

In [11]:
basic_model, basic_test_loader = run_experiment_for_fairness("", feature_cols, demographics=False)

In [12]:
# ============================================================
# BASIC MODEL
# ============================================================

basic_probs, basic_preds, basic_targets = get_predictions_basic(
    basic_model,
    basic_test_loader,
    device
)

_, _, test_users = create_sequences(
    test_df,
    feature_cols,
    SEQUENCE_LENGTH
)

basic_fairness_df = create_fairness_df_from_users(
    test_users,
    users_clean
)

basic_fairness_df["y_true"] = basic_targets
basic_fairness_df["y_pred"] = basic_preds

print("\n")
print("==================================================")
print("BASIC MODEL FAIRNESS")
print("==================================================")







BASIC MODEL FAIRNESS


In [13]:
# -----------------------------
# Gender fairness
# -----------------------------

gender_metrics_basic = compute_group_metrics(
    basic_fairness_df,
    "gender"
)

print("\nGender metrics")
display(gender_metrics_basic)

compute_parity_gaps(gender_metrics_basic, "gender")





Gender metrics


,gender,n_samples,TPR,FPR,PPV,NPV
0,FEMALE,16631,0.893265,0.238623,0.775141,0.885664
1,MALE,10822,0.887828,0.254698,0.779371,0.867663
2,Other,1476,0.900744,0.186567,0.853114,0.872000
3,Unknown,3442,0.878031,0.318137,0.654787,0.890525



===== FAIRNESS GAPS (gender) =====
TPR_gap: 0.0227
FPR_gap: 0.1316
PPV_gap: 0.1983
NPV_gap: 0.0229


{'TPR_gap': 0.022713033135931204,
 'FPR_gap': 0.13157009072285628,
 'PPV_gap': 0.19832674950621287,
 'NPV_gap': 0.02286198336770784}

In [14]:
# -----------------------------
# Canton fairness
# -----------------------------

canton_metrics_basic = compute_group_metrics(
    basic_fairness_df,
    "canton"
)

print("\nCanton metrics")
display(canton_metrics_basic)

compute_parity_gaps(canton_metrics_basic, "canton")


Canton metrics


,canton,n_samples,TPR,FPR,PPV,NPV
0,AG,3942,0.911624,0.233566,0.872713,0.831563
1,AR,807,0.666667,0.092593,0.518519,0.947917
2,Andere,223,0.926829,0.250000,0.820144,0.892857
3,BE,1630,0.908981,0.235732,0.797657,0.891462
4,BL,1361,0.852632,0.280172,0.744094,0.836394
5,BS,1290,0.944371,0.095327,0.933246,0.920152
6,FL,316,0.907407,0.103896,0.901840,0.901961
7,FR,12,NaN,0.500000,0.000000,1.000000
8,LU,33,0.736842,0.214286,0.823529,0.687500
9,SG,9350,0.895030,0.246578,0.725889,0.907733



===== FAIRNESS GAPS (canton) =====
TPR_gap: 0.2777
FPR_gap: 0.5000
PPV_gap: 0.9332
NPV_gap: 0.3125


{'TPR_gap': 0.27770419426048565,
 'FPR_gap': 0.5,
 'PPV_gap': 0.9332460732984293,
 'NPV_gap': 0.3125}

### Model with demographic features

In [16]:
demo_model, test_loader_demo = run_experiment_for_fairness("", feature_cols, demographics=True)

In [18]:
# ============================================================
# DEMOGRAPHIC MODEL
# ============================================================

demo_probs, demo_preds, demo_targets = get_predictions_demo(
    demo_model,
    test_loader_demo,
    device
)

_, _, test_users = create_sequences(
    test_df,
    feature_cols,
    SEQUENCE_LENGTH
)

demo_fairness_df = create_fairness_df_from_users(
    test_users,
    users_clean
)


demo_fairness_df["y_true"] = demo_targets
demo_fairness_df["y_pred"] = demo_preds

In [19]:
# -----------------------------
# Gender fairness
# -----------------------------

gender_metrics_demo = compute_group_metrics(
    demo_fairness_df,
    "gender"
)

print("\nGender metrics")
display(gender_metrics_demo)

compute_parity_gaps(gender_metrics_demo, "gender")



Gender metrics


,gender,n_samples,TPR,FPR,PPV,NPV
0,FEMALE,16631,0.874200,0.189651,0.809336,0.874922
1,MALE,10822,0.880117,0.194047,0.821312,0.869007
2,Other,1476,0.893300,0.119403,0.900000,0.872781
3,Unknown,3442,0.792439,0.217157,0.714929,0.845869



===== FAIRNESS GAPS (gender) =====
TPR_gap: 0.1009
FPR_gap: 0.0978
PPV_gap: 0.1851
NPV_gap: 0.0291


{'TPR_gap': 0.10086087581370817,
 'FPR_gap': 0.09775387767047118,
 'PPV_gap': 0.18507078507078512,
 'NPV_gap': 0.029053416039448665}

In [20]:
# -----------------------------
# Canton fairness
# -----------------------------

canton_metrics_demo = compute_group_metrics(
    demo_fairness_df,
    "canton"
)

print("\nCanton metrics")
display(canton_metrics_demo)

compute_parity_gaps(canton_metrics_demo, "canton")


Canton metrics


,canton,n_samples,TPR,FPR,PPV,NPV
0,AG,3942,0.944268,0.256643,0.866009,0.883624
1,AR,807,0.580952,0.039886,0.685393,0.938719
2,Andere,223,0.853659,0.260000,0.801527,0.804348
3,BE,1630,0.906553,0.217122,0.810195,0.891243
4,BL,1361,0.905263,0.186782,0.822404,0.899841
5,BS,1290,0.957616,0.115888,0.921019,0.936634
6,FL,316,0.901235,0.103896,0.901235,0.896104
7,FR,12,NaN,0.000000,NaN,1.000000
8,LU,33,0.631579,0.357143,0.705882,0.562500
9,SG,9350,0.841278,0.178320,0.774872,0.876480



===== FAIRNESS GAPS (canton) =====
TPR_gap: 0.3767
FPR_gap: 0.3571
PPV_gap: 0.2356
NPV_gap: 0.4375


{'TPR_gap': 0.37666351308735413,
 'FPR_gap': 0.35714285714285715,
 'PPV_gap': 0.2356258498532885,
 'NPV_gap': 0.4375}